In [16]:
from pathlib import Path
import sys
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import (
    MultiModalRawDataset,
    multimodal_raw_collate_fn,
)
from src.models.multimodal.multimodal_text_guided_pvd_classifier import (
    MultimodalTextGuidedPVDClassifier,
)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Config

In [3]:
IMAGE_ROOT = PROJECT_ROOT / "data" / "AIDG" / "dataset_PlantDoc" / "images" / "train"
CAPTION_ROOT = PROJECT_ROOT / "data" / "AIDG" / "captions_LLaVA" / "train"

NUM_CLASSES = 28
BATCH_SIZE = 8
EPOCHS = 10
LR = 1e-4
NUM_WORKERS = 2
IMG_SIZE = 224

USE_DEPTH_SUPPRESSED = False
STRICT_CAPTION_MATCH = True

SAVE_DIR = PROJECT_ROOT / "archive" / "multimodal_text_guided"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = SAVE_DIR / "multimodal_text_guided_pvd_classifier_state_dict.pth"
CKPT_PATH = SAVE_DIR / "multimodal_text_guided_pvd_classifier_checkpoint.pth"
CONFIG_PATH = SAVE_DIR / "train_config.json"

Transform

In [4]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================
# DATASET / LOADER
# =========================
dataset = MultiModalRawDataset(
    image_root=IMAGE_ROOT,
    caption_root=CAPTION_ROOT,
    transform=transform,
    use_depth_suppressed=USE_DEPTH_SUPPRESSED,
    strict_caption_match=STRICT_CAPTION_MATCH
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=multimodal_raw_collate_fn,
    pin_memory=(device == "cuda")
)

[MultiModalRawDataset] Total selected images: 2336
[MultiModalRawDataset] Valid samples: 2336
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf': 25, 'grape_leaf': 26, 'grape_leaf_black_rot': 27}


Model

In [5]:
model = MultimodalTextGuidedPVDClassifier(
    num_classes=NUM_CLASSES,
    clip_model_name="ViT-L-14",
    clip_pretrained="openai",
    text_input_dim=768,
    image_input_dim=1024,
    proj_dim=512,
    proj_hidden_dim=768,
    pvd_hidden_dim=768,
    cls_hidden_dim=1024,
    dropout=0.3,
    normalize_projection=False,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Save config

In [6]:
train_config = {
    "image_root": str(IMAGE_ROOT),
    "caption_root": str(CAPTION_ROOT),
    "num_classes": NUM_CLASSES,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "num_workers": NUM_WORKERS,
    "img_size": IMG_SIZE,
    "use_depth_suppressed": USE_DEPTH_SUPPRESSED,
    "strict_caption_match": STRICT_CAPTION_MATCH,
    "device": device,
    "model": {
        "clip_model_name": "ViT-L-14",
        "clip_pretrained": "openai",
        "text_input_dim": 768,
        "image_input_dim": 1024,
        "proj_dim": 512,
        "proj_hidden_dim": 768,
        "pvd_hidden_dim": 768,
        "cls_hidden_dim": 1024,
        "dropout": 0.3,
        "normalize_projection": False
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(train_config, f, indent=2, ensure_ascii=False)

print("Saved config to:", CONFIG_PATH)
print("Using device:", device)
print("Dataset size:", len(dataset))
print("Model save path:", MODEL_PATH)

Saved config to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\train_config.json
Using device: cpu
Dataset size: 2336
Model save path: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_state_dict.pth


Sanity check

In [7]:
print("\nRunning sanity check on 1 batch...")
batch = next(iter(loader))
images = batch["image"].to(device)
texts = batch["text"]
labels = batch["label"].to(device)

with torch.no_grad():
    logits = model(images, texts)

print("images.shape:", images.shape)
print("len(texts):", len(texts))
print("labels.shape:", labels.shape)
print("logits.shape:", logits.shape)
print("Sanity check passed.\n")


Running sanity check on 1 batch...
images.shape: torch.Size([8, 3, 224, 224])
len(texts): 8
labels.shape: torch.Size([8])
logits.shape: torch.Size([8, 28])
Sanity check passed.



Train loop

In [8]:
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        texts = batch["text"]
        labels = batch["label"].to(device, non_blocking=True)

        logits = model(images, texts)   # [B, num_classes]
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")

    # save best by training acc (tạm thời)
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), MODEL_PATH)

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_acc": best_acc,
            "config": train_config
        }
        torch.save(checkpoint, CKPT_PATH)

        print(f"Saved best model to: {MODEL_PATH}")
        print(f"Saved checkpoint to: {CKPT_PATH}")

print("\nTraining finished.")
print("Best training acc:", best_acc)

Epoch [1/10] - Loss: 1.9077 - Acc: 0.4174
Saved best model to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_state_dict.pth
Saved checkpoint to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_checkpoint.pth
Epoch [2/10] - Loss: 0.9694 - Acc: 0.6580
Saved best model to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_state_dict.pth
Saved checkpoint to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_checkpoint.pth
Epoch [3/10] - Loss: 0.6694 - Acc: 0.7628
Saved best model to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_state_dict.pth
Saved checkpoint to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_checkpoint.pt

OSError: Caught OSError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\_utils\worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "G:\My Drive\Documents\GR1\MulCo-PlantNet\src\datasets\multimodal_raw_dataset.py", line 213, in __getitem__
    image = Image.open(item["image_path"]).convert("RGB")
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py", line 3480, in open
    prefix = fp.read(16)
             ^^^^^^^^^^^
OSError: [Errno 22] Invalid argument


# Resume training

Import + path

In [29]:
from pathlib import Path
import sys
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

def find_project_root(start_path: Path, required_dir: str = "src") -> Path:
    current = start_path.resolve()
    while True:
        if (current / required_dir).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Cannot find project root containing '{required_dir}'")
        current = current.parent

PROJECT_ROOT = find_project_root(Path.cwd(), required_dir="src")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import (
    MultiModalRawDataset,
    multimodal_raw_collate_fn,
)
from src.models.multimodal.multimodal_text_guided_pvd_classifier import (
    MultimodalTextGuidedPVDClassifier,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Imports OK")

PROJECT_ROOT: G:\My Drive\Documents\GR1\MulCo-PlantNet
Imports OK


Config resume

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_ROOT = PROJECT_ROOT / "data" / "AIDG" / "dataset_PlantDoc" / "images" / "train"
CAPTION_ROOT = PROJECT_ROOT / "data" / "AIDG" / "captions_LLaVA" / "train"

NUM_CLASSES = 28
BATCH_SIZE = 8
IMG_SIZE = 224
LR = 1e-4
NUM_WORKERS = 0   # QUAN TRỌNG: đổi về 0 để tránh lỗi DataLoader worker trên Windows
USE_DEPTH_SUPPRESSED = False
STRICT_CAPTION_MATCH = True

SAVE_DIR = PROJECT_ROOT / "archive" / "multimodal_text_guided"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = SAVE_DIR / "multimodal_text_guided_pvd_classifier_state_dict.pth"
CKPT_PATH = SAVE_DIR / "multimodal_text_guided_pvd_classifier_checkpoint.pth"
CONFIG_PATH = SAVE_DIR / "train_config.json"

# Train tiếp thêm bao nhiêu epoch nữa
EXTRA_EPOCHS = 6

print("device:", device)
print("checkpoint exists:", CKPT_PATH.exists())
print("model exists:", MODEL_PATH.exists())

device: cpu
checkpoint exists: True
model exists: True


Transform + dataset

In [31]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = MultiModalRawDataset(
    image_root=IMAGE_ROOT,
    caption_root=CAPTION_ROOT,
    transform=transform,
    use_depth_suppressed=USE_DEPTH_SUPPRESSED,
    strict_caption_match=STRICT_CAPTION_MATCH
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,   # ép 0
    collate_fn=multimodal_raw_collate_fn,
    pin_memory=(device == "cuda")
)

print("Dataset size:", len(dataset))
print("Num batches:", len(loader))

[MultiModalRawDataset] Total selected images: 2336
[MultiModalRawDataset] Valid samples: 2336
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf': 25, 'grape_leaf': 26, 'grape_leaf_black_rot': 27}


model ver2 + optimizer + load checkpoint

In [33]:
model = MultimodalTextGuidedPVDClassifier(
    num_classes=NUM_CLASSES,
    clip_model_name="ViT-L-14",
    clip_pretrained="openai",
    text_input_dim=768,
    image_input_dim=1024,
    proj_dim=512,
    proj_hidden_dim=768,
    pvd_hidden_dim=768,
    cls_hidden_dim=1024,
    dropout=0.3,
    normalize_projection=False,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss()

# =====================================================
# Freeze ConvNeXt backbone, keep TG-CBAM + head trainable
# =====================================================
for param in model.image_encoder.model.parameters():
    param.requires_grad = False

for module in [
    model.image_encoder.tgcbam1,
    model.image_encoder.tgcbam2,
    model.image_encoder.tgcbam3,
    model.image_encoder.tgcbam4,
]:
    for param in module.parameters():
        param.requires_grad = True

for param in model.image_encoder.norm4.parameters():
    param.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=LR)

checkpoint = torch.load(CKPT_PATH, map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    raise KeyError("Checkpoint does not contain 'model_state_dict'")

# Nếu load optimizer state từ checkpoint cũ bị mismatch do số params đã đổi,
# ta nên bỏ qua và dùng optimizer mới
try:
    if "optimizer_state_dict" in checkpoint and checkpoint["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        print("optimizer state loaded")
except Exception as e:
    print("[WARN] Could not load optimizer state from old checkpoint.")
    print("[WARN] Reason:", e)
    print("[WARN] Using fresh optimizer because trainable parameter groups changed.")

start_epoch = checkpoint.get("epoch", 0) + 1
best_acc = checkpoint.get("best_acc", 0.0)
history = checkpoint.get("history", [])

# end_epoch = start_epoch + EXTRA_EPOCHS - 1
end_epoch = 10

total_params = sum(p.numel() for p in model.parameters())
trainable_params_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Resuming from epoch:", start_epoch)
print("Training until epoch:", end_epoch)
print("Best acc so far:", best_acc)
print("Total params:", total_params)
print("Trainable params:", trainable_params_count)
print("Frozen params:", total_params - trainable_params_count)
print("Trainable ratio:", trainable_params_count / total_params)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[WARN] Could not load optimizer state from old checkpoint.
[WARN] Reason: loaded state dict contains a parameter group that doesn't match the size of optimizer's group
[WARN] Using fresh optimizer because trainable parameter groups changed.
Resuming from epoch: 8
Training until epoch: 10
Best acc so far: 0.9113869863013698
Total params: 522644741
Trainable params: 6436764
Frozen params: 516207977
Trainable ratio: 0.012315753886060819


train one epoch

In [34]:
def train_one_epoch_ver2(model, loader, criterion, optimizer, device="cpu"):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch_idx, batch in enumerate(progress_bar):
        images = batch["image"].to(device, non_blocking=True)
        texts = batch["text"]
        labels = batch["label"].to(device, non_blocking=True)

        logits = model(images, texts)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(logits, dim=1)

        batch_size_actual = labels.size(0)
        total += batch_size_actual
        correct += (preds == labels).sum().item()
        running_loss += loss.item() * batch_size_actual

        progress_bar.set_postfix({
            "loss": f"{running_loss / max(total, 1):.4f}",
            "acc": f"{correct / max(total, 1):.4f}",
        })

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

Resume training loop

In [35]:
for epoch in range(start_epoch, end_epoch + 1):
    epoch_loss, epoch_acc = train_one_epoch_ver2(
        model=model,
        loader=loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    print(f"Epoch [{epoch}/{end_epoch}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")

    history.append({
        "epoch": epoch,
        "loss": epoch_loss,
        "acc": epoch_acc,
        "lr": optimizer.param_groups[0]["lr"],
    })

    # save best model
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Saved best model to: {MODEL_PATH}")

    # save checkpoint
    checkpoint_to_save = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_acc": best_acc,
        "history": history,
    }
    torch.save(checkpoint_to_save, CKPT_PATH)
    print(f"Saved checkpoint to: {CKPT_PATH}")

Training:   0%|          | 0/292 [00:00<?, ?it/s]

Epoch [8/10] - Loss: 0.1387 - Acc: 0.9563
Saved best model to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_state_dict.pth
Saved checkpoint to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_checkpoint.pth


KeyboardInterrupt: 

Save resumed config

In [ ]:
resume_config = {
    "image_root": str(IMAGE_ROOT),
    "caption_root": str(CAPTION_ROOT),
    "num_classes": NUM_CLASSES,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "num_workers": NUM_WORKERS,
    "img_size": IMG_SIZE,
    "use_depth_suppressed": USE_DEPTH_SUPPRESSED,
    "strict_caption_match": STRICT_CAPTION_MATCH,
    "device": device,
    "resume_from": str(CKPT_PATH),
    "start_epoch": start_epoch,
    "end_epoch": end_epoch,
    "best_acc_before_resume": best_acc,
    "model": {
        "clip_model_name": "ViT-L-14",
        "clip_pretrained": "openai",
        "text_input_dim": 768,
        "image_input_dim": 1024,
        "proj_dim": 512,
        "proj_hidden_dim": 768,
        "pvd_hidden_dim": 768,
        "cls_hidden_dim": 1024,
        "dropout": 0.3,
        "normalize_projection": False
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(resume_config, f, indent=2, ensure_ascii=False)

print("Saved updated config to:", CONFIG_PATH)